In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
%matplotlib inline

In [ ]:
df = pd.read_csv("../data/novelty_peeking.csv")

# Считаем дневную конверсию по группам
daily_cr = df.groupby(["day", "group"])["purchased"].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=daily_cr, x="day", y="purchased", hue="group", marker="o")
plt.title("Динамика конверсии по дням (Novelty Effect)", fontsize=14)
plt.ylabel("Conversion Rate")
plt.xlabel("День теста")
plt.axvline(x=3.5, color='red', linestyle='--', label='Конец эффекта новизны')
plt.legend()
plt.show()

In [ ]:
results = []

# Накапливаем данные день за днем
for day in range(1, 15):
    current_data = df[df["day"] <= day]
    
    control = current_data[current_data["group"] == "control"]
    treatment = current_data[current_data["group"] == "treatment"]
    
    conv_c = control["purchased"].sum()
    conv_t = treatment["purchased"].sum()
    n_c = len(control)
    n_t = len(treatment)
    
    # Z-test
    z_stat, p_value = proportions_ztest([conv_t, conv_c], [n_t, n_c])
    
    # Считаем общую конверсию
    cr_c = conv_c / n_c
    cr_t = conv_t / n_t
    
    results.append({
        "day": day,
        "p_value": p_value,
        "cr_control": cr_c,
        "cr_treatment": cr_t,
        "significant": p_value < 0.05
    })

peek_df = pd.DataFrame(results)
peek_df

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(peek_df["day"], peek_df["p_value"], marker="o", color="purple")
plt.axhline(y=0.05, color="red", linestyle="--", label="Порог значимости (0.05)")

# Красим точки: зеленые, если значимо
colors = ['green' if p < 0.05 else 'gray' for p in peek_df["p_value"]]
plt.scatter(peek_df["day"], peek_df["p_value"], c=colors, s=100, zorder=5)

plt.title("p-value по дням (Симуляция Peeking)", fontsize=14)
plt.ylabel("p-value")
plt.xlabel("День теста")
plt.legend()
plt.show()

In [ ]:
final_data = df

control = final_data[final_data["group"] == "control"]
treatment = final_data[final_data["group"] == "treatment"]

conv_c = control["purchased"].sum()
conv_t = treatment["purchased"].sum()
n_c = len(control)
n_t = len(treatment)

cr_c = conv_c / n_c
cr_t = conv_t / n_t

z_stat, p_value = proportions_ztest([conv_t, conv_c], [n_t, n_c])

print(f"ИТОГ ЗА 14 ДНЕЙ:")
print(f"Control CR: {cr_c:.2%}")
print(f"Treatment CR: {cr_t:.2%}")
print(f"p-value: {p_value:.4f}")